# **Midterm Project**

## Data Preparation

In [ ]:
import pandas as pd

# Load daily stock data to dataframes
universe = ['AAPL','AMZN','GOOG','IBM','META','MSFT','NFLX','ORCL','SAP','TSLA']
data = {}

for ticker in universe:
    df = pd.read_csv(f'{ticker}.csv', index_col=0)
    df.index = pd.to_datetime(df.index, format='%d-%b-%y', errors='coerce')
    data[ticker] = df

              Open    High    Low   Close  Adj Close       Volume
Date                                                             
2018-12-28  102.09  102.41  99.52  100.39      94.81   38,196,300
2018-12-27   99.30  101.19  96.40  101.18      95.56   49,498,500
2018-12-26   95.14  100.69  93.96  100.56      94.97   51,634,800
2018-12-24   97.68   97.97  93.98   94.13      88.90   43,935,200
2018-12-21  101.63  103.00  97.46   98.23      92.77  111,242,100


## Close Data

In [ ]:
# Extract close and adj close data
close_data = pd.DataFrame()

for ticker, df in data.items():
    close_data[f'{ticker} Close'] = df['Close']
    close_data[f'{ticker} Adj Close'] = df['Adj Close']

# Extra precaution
close_data.sort_index(inplace=True)

print(close_data.head())

            AAPL Close  AAPL Adj Close  AMZN Close  AMZN Adj Close  \
Date                                                                 
2018-01-02       43.06           40.57       59.45           59.45   
2018-01-03       43.06           40.56       60.21           60.21   
2018-01-04       43.26           40.75       60.48           60.48   
2018-01-05       43.75           41.21       61.46           61.46   
2018-01-08       43.59           41.06       62.34           62.34   

            GOOG Close  GOOG Adj Close  IBM Close  IBM Adj Close  META Close  \
Date                                                                           
2018-01-02       53.25           53.12     147.47         107.53      181.42   
2018-01-03       54.12           53.99     151.52         110.49      184.67   
2018-01-04       54.32           54.19     154.59         112.72      184.33   
2018-01-05       55.11           54.98     155.34         113.27      186.85   
2018-01-08       55.35       

## 1.1 Portfolio set-up

In [ ]:
class Portfolio:
    def __init__(self, tickers, initial_cash=5e6, start_date='2018-01-01'):
        self.cash = initial_cash
        self.tickers = tickers
        self.start_date = pd.to_datetime(start_date)
        
        # each ticker has its own transaction DataFrame
        self.holdings = {
            t: pd.DataFrame([{
                'Buy/Sell': 0,
                'Stock Price': 0,
                'Delta Shares': 0,
                'Current Holdings': 0,
                'Current Value': 0
            }], index=[self.start_date])
            for t in tickers
        }
        
    def buy(self, ticker, trade_date, price, shares):
        cost = price * shares
        if cost > self.cash:
            print(f"⚠️ Not enough cash to buy {shares} shares of {ticker}")
            return
        self.cash -= cost
        prev = self.holdings[ticker].iloc[-1]['Current Holdings']
        new = prev + shares
        self.holdings[ticker].loc[pd.to_datetime(trade_date)] = {
            'Buy/Sell': 1,
            'Stock Price': price,
            'Delta Shares': shares,
            'Current Holdings': new,
            'Current Value': new * price
        }

    def sell(self, ticker, trade_date, price, shares):
        prev = self.holdings[ticker].iloc[-1]['Current Holdings']
        if shares > prev:
            print(f"⚠️ Not enough shares to sell {shares} of {ticker}")
            return
        proceeds = price * shares
        self.cash += proceeds
        new = prev - shares
        self.holdings[ticker].loc[pd.to_datetime(trade_date)] = {
            'Buy/Sell': -1,
            'Stock Price': price,
            'Delta Shares': -shares,
            'Current Holdings': new,
            'Current Value': new * price
        }

    def get_value(self, prices, date):
        """Compute total mark-to-market value given price data and date."""
        total_value = self.cash
        for t in self.tickers:
            if date in prices.index:
                px = prices.loc[date, f'{t} Close']
                shares = self.holdings[t].iloc[-1]['Current Holdings']
                total_value += shares * px
        return total_value
    
    def mtm(self):
        """
        Compute total mark-to-market portfolio value using the last known
        stock price for each holding (from the portfolio DataFrames) + cash.
        
        Returns:
            float : total portfolio value in USD
        """
        total_value = self.cash
        for ticker in self.tickers:
            df = self.holdings[ticker]
            if not df.empty:
                last_row = df.iloc[-1]
                shares = last_row['Current Holdings']
                price = last_row['Stock Price']
                total_value += shares * price
        return total_value



## 1.2 Initial trades

In [51]:
pyport = Portfolio(universe)
allocation = 1e6
buy_list = ['IBM','MSFT','GOOG','AAPL','AMZN']
trade_date = '2018-01-02'

for ticker in buy_list:
    price = close_data.loc[trade_date, f'{ticker} Close']
    shares = int(allocation // price)
    pyport.buy(ticker, trade_date, price, shares)

print(f"Remaining cash: ${pyport.cash:,.2f}")
print(pyport.holdings['MSFT'].tail())


Remaining cash: $150.50
            Buy/Sell  Stock Price  Delta Shares  Current Holdings  \
2018-01-01         0         0.00             0                 0   
2018-01-02         1        85.95         11634             11634   

            Current Value  
2018-01-01            0.0  
2018-01-02       999942.3  


In [ ]:
current_value = pyport.mtm()
print(f"Total portfolio value (using last known prices): ${current_value:,.2f}")
